# Market Overview feature feasibility audit — 28 Sep 2026

Research question: which requested market visualizations can be computed faithfully from the local, approved Sectors.app snapshots, and which lack required grain/fields? This is a descriptive data-quality audit, not investment advice or a validated signal.

Roadmap: inspect source snapshots → verify grain and coverage → test RSI lookback feasibility → document intraday-flow and MVRV gaps → conclude with non-fabricated UI states. All market data stays local; no API calls are made.

## Runtime and source guard

Run from the Hackathon Sectors project folder. For Colab/Kaggle, attach the project data artifact containing `data/raw/sectors/` and the read-only loader at `src/idx_evidence_lab/market_data.py`; the notebook stops with a direct message if either is absent. Only existing local Sectors.app snapshots are used.

In [1]:
from pathlib import Path
import sys, json, glob, collections, platform

candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/Hackathon Sectors')]
ROOT = next((p for p in candidates if (p / 'data/raw/sectors').is_dir() and (p / 'src/idx_evidence_lab/market_data.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Attach/copy the Hackathon Sectors project artifact containing data/raw/sectors and src/idx_evidence_lab/market_data.py, then run again.')
sys.path.insert(0, str(ROOT / 'src'))
from idx_evidence_lab.market_data import load_ihsg_snapshot, load_lq45_universe, load_issuer_daily
print('Python:', platform.python_version())
print('Project root:', ROOT)
print('Source policy: local Sectors.app snapshots only; no network calls')

Python: 3.14.2
Project root: /Users/rasyaaudreabramantyawijaya/Downloads/Hackathon Sectors
Source policy: local Sectors.app snapshots only; no network calls


## IHSG and daily LQ45 data profile

The unit of observation is one completed daily close per index/ticker. RSI is computable from this grain. Current constituent membership has a separate checksum issue, so the issuer set should remain provisional until that snapshot hash is reconciled.

In [2]:
ihsg = load_ihsg_snapshot(ROOT)
universe = load_lq45_universe(ROOT)
symbols = [row['ticker'] for row in universe.get('symbols', [])]
daily = {symbol: load_issuer_daily(ROOT, symbol) for symbol in symbols}
counts = sorted(x['observation_count'] for x in daily.values() if x['ok'])
print('IHSG:', ihsg['data_quality'], ihsg['observation_count'], ihsg['coverage_start'], '→', ihsg['coverage_end'])
print('Universe:', universe['data_quality'], universe['symbol_count'], 'symbols; issues:', universe['issues'][:2])
print('Daily OHLCV:', sum(x['ok'] for x in daily.values()), '/', len(symbols), 'tickers; verified:', sum(x['data_quality']=='VERIFIED' for x in daily.values()))
print('Daily rows min / median / max:', counts[0], counts[len(counts)//2], counts[-1])
print('Daily coverage:', min(x['coverage_start'] for x in daily.values() if x['ok']), '→', max(x['coverage_end'] for x in daily.values() if x['ok']))

IHSG: VERIFIED 708 2023-09-25 → 2026-09-24
Universe: PARTIAL 45 symbols; issues: ['Universe snapshot hash is missing or mismatched.']
Daily OHLCV: 45 / 45 tickers; verified: 45
Daily rows min / median / max: 424 712 712
Daily coverage: 2023-09-25 → 2026-09-24


## Broker-flow grain audit

The local archive has four broker codes and daily date keys. Daily aggregates cannot be split into hourly buy/sell sessions without inventing intraday allocation. The previous project audit treats this archive as supplementary, not as a substitute for issuer-level broker-flow snapshots.

In [3]:
broker_files = glob.glob(str(ROOT / 'data/raw/sectors/cache_lq45/broker_flow/*.json'))
broker_codes, dates, payload_fields = set(), set(), set()
for filename in broker_files:
    payload = json.loads(Path(filename).read_text())
    body = payload.get('body', {})
    broker_codes.add(body.get('broker_code'))
    payload_fields.update(body.keys())
    for day in body.get('data', []):
        dates.add(day.get('date'))
        for row in day.get('summary', []): payload_fields.update(row.keys())
print('Broker archive files:', len(broker_files), '| codes:', sorted(broker_codes))
print('Date-key range:', min(dates), '→', max(dates), '| rows have calendar date but no session timestamp field')
print('Buy/sell value fields present:', {'bval','sval','nval'}.issubset(payload_fields))
print('Intraday time/session fields present:', sorted(x for x in payload_fields if any(k in x.lower() for k in ['time','hour','session','timestamp'])))
print('Interpretation: daily broker summary exists in a four-code archive; issuer-level adapter/provenance and hour-by-hour series remain unresolved.')

Broker archive files: 228 | codes: ['AK', 'XL', 'YP', 'ZP']
Date-key range: 2025-01-02 → 2026-09-11 | rows have calendar date but no session timestamp field
Buy/sell value fields present: True
Intraday time/session fields present: []
Interpretation: daily broker summary exists in a four-code archive; issuer-level adapter/provenance and hour-by-hour series remain unresolved.


## RSI breadth by lookback

Formula: Wilder RSI from close-to-close changes, with Wilder-smoothed average gains/losses. Five provisional bands are Overbought ≥70; Strong 60–<70; Neutral 40–<60; Weak 30–<40; Oversold <30. These descriptive bins do not imply trading instructions. A longer lookback smooths the series and may compress cross-sectional differences during quiet markets; exact RSI values should remain available alongside category counts.

In [4]:
def wilder_rsi(closes, period):
    if len(closes) <= period: return None
    changes = [closes[i] - closes[i-1] for i in range(1, len(closes))]
    gain = sum(max(x, 0) for x in changes[:period]) / period
    loss = sum(max(-x, 0) for x in changes[:period]) / period
    for change in changes[period:]:
        gain = (gain*(period-1) + max(change,0)) / period
        loss = (loss*(period-1) + max(-change,0)) / period
    if gain == loss == 0: return 50.0
    if loss == 0: return 100.0
    return 100 - 100/(1 + gain/loss)

def band(value):
    return 'Overbought' if value >= 70 else 'Strong' if value >= 60 else 'Neutral' if value >= 40 else 'Weak' if value >= 30 else 'Oversold'

periods = [14, 21, 30, 50, 100, 200]
band_names = ['Overbought','Strong','Neutral','Weak','Oversold']
results = {}
for period in periods:
    values = {}
    for symbol, item in daily.items():
        if item['ok']:
            closes = [float(row['close']) for row in item['series']]
            value = wilder_rsi(closes, period)
            if value is not None: values[symbol] = value
    results[period] = values
    counts = collections.Counter(band(value) for value in values.values())
    print(f'RSI({period:>3}) · n={len(values):>2}/45 | ' + ' | '.join(f'{name}: {counts[name]}' for name in band_names) + f' | range {min(values.values()):.1f}–{max(values.values()):.1f}')

RSI( 14) · n=45/45 | Overbought: 0 | Strong: 1 | Neutral: 27 | Weak: 15 | Oversold: 2 | range 25.6–66.1
RSI( 21) · n=45/45 | Overbought: 0 | Strong: 1 | Neutral: 37 | Weak: 7 | Oversold: 0 | range 32.0–66.4
RSI( 30) · n=45/45 | Overbought: 0 | Strong: 1 | Neutral: 42 | Weak: 2 | Oversold: 0 | range 35.6–64.3
RSI( 50) · n=45/45 | Overbought: 0 | Strong: 0 | Neutral: 44 | Weak: 1 | Oversold: 0 | range 39.0–60.0
RSI(100) · n=45/45 | Overbought: 0 | Strong: 0 | Neutral: 45 | Weak: 0 | Oversold: 0 | range 43.3–55.7
RSI(200) · n=45/45 | Overbought: 0 | Strong: 0 | Neutral: 45 | Weak: 0 | Oversold: 0 | range 45.3–53.7


## Updated feature conclusion: foreign flow and valuation

MVRV is not a direct equity-index metric. The user has replaced it with forward P/E Z-band for LQ45. The local source contains only a single forward P/E snapshot (36 positive values among 45 reports), no historical forward P/E index series, and no LQ45 index-level price history. Therefore the Z-band is not calculated. The currently pulled LQ45 data is its 45-issuer universe and constituent daily prices, not the index's own historical series.

Daily foreign buy/sell values are available from Sectors.app foreign-flow snapshots and can be displayed as stacked daily bars over a selected 1–12 month trailing range, but must not be called all-broker transaction volume. A descriptive unusual *foreign* flow count can be calculated per issuer from a past-only 20-session baseline; issuer-level broker unusual flow remains unresolved. The four-broker archive is not a valid total-LQ45 substitute.

A monthly foreign net-flow Z-score uses 12 prior full months only and excludes the open current month. A value below −2 is an extreme-outflow research hypothesis, **not** a validated buy signal. The claim that it coincides with LQ45 bottoms or is highly accurate needs point-in-time constituents, a historical LQ45 index series, an event protocol, and out-of-sample false-positive analysis.

The cross-sectional aggregate market-cap / FY2025 operating-cash-flow ratio is calculable from the current report snapshot (45/45 field coverage) but mixes Sep 2026 capitalizations with FY2025 annual cash flows, including banks. It is not MVRV and cannot become a historical Z-band without aligned time-series observations. No external API call was made.

## Reproducible offline calculations for the revised panels

The loader verifies each snapshot hash, Sectors.app endpoint and HTTP status, validates buy − sell = net, and deduplicates by ticker/date. The ratio below is descriptive rather than an index valuation series.

In [ ]:
from idx_evidence_lab.market_overview_analytics import load_foreign_flow, load_valuation_snapshot
flow = load_foreign_flow(ROOT, symbols)
valuation = load_valuation_snapshot(ROOT, symbols)
print('Foreign daily quality/range:', flow['data_quality'], len(flow['daily']), flow['coverage_start'], flow['coverage_end'])
print('Issuer coverage per day:', min(row['coverage'] for row in flow['daily']), max(row['coverage'] for row in flow['daily']))
print('Unusual foreign flow, latest day:', flow['unusual_foreign_flow'])
print('Monthly Z, complete months only:', [(row['month'], round(row['z12'], 2)) for row in flow['monthly'] if row['z12'] is not None])
print('Forward P/E report coverage:', valuation['forward_pe_coverage'], '/', valuation['universe_count'])
print('Cap/FY2025 OCF snapshot ratio:', valuation['cap_to_ocf'])
assert all(row['z12'] is None for row in flow['monthly'] if not row['complete'])
assert valuation['lq45_index_history_pulled'] is False

## Dashboard summary view — 2 October 2026

Presentation-only update: reuse the same approved local inputs and formulas, without new models or external fetches. The four small views answer: which foreign flows are unusual, how broad is participation, what is the IHSG trend context, and what is the current LQ45 capitalization? Detail remains in Screener/Market Overview. Breadth counts shares, flow measures foreign transactions, regime is a descriptive technical rule, and capitalization is a single report snapshot—not the whole IDX or a historical curve. Attach the exported market-overview-data.json for Colab/Kaggle. This cell reconstructs the summary and displays its coverage/dates. No training or split is needed for this presentation-only view.


In [ ]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# Attach the exported bundle in Colab/Kaggle; no API calls.
candidates = [Path("market-overview-data.json"), Path("docs/prototypes/market-overview-data.json"), Path("/content/market-overview-data.json")]
if Path("/kaggle/input").exists():
    candidates += list(Path("/kaggle/input").rglob("market-overview-data.json"))
bundle_path = next((p for p in candidates if p.is_file()), None)
if bundle_path is None:
    raise FileNotFoundError("Attach docs/prototypes/market-overview-data.json from the project.")
bundle = json.loads(bundle_path.read_text())
assert bundle["provider"] == "Sectors.app"
dates = [r["date"] for r in bundle["lq45_index"]["series"]][-2:]
changes = []
above50 = []
for source in bundle["lq45_daily"].values():
    if not source["ok"]:
        continue
    rows = {r["date"]: r["close"] for r in source["series"] if r["close"] > 0}
    if all(date in rows for date in dates):
        changes.append(rows[dates[1]] / rows[dates[0]] - 1)
    history = [v for d, v in sorted(rows.items()) if d <= dates[-1]][-50:]
    if dates[-1] in rows and len(history) == 50:
        above50.append(rows[dates[-1]] > np.mean(history))
counts = [sum(r > 0 for r in changes), sum(r == 0 for r in changes), sum(r < 0 for r in changes)]
assert sum(counts) == len(changes)
prices = np.array([r["price"] for r in bundle["ihsg"]["series"]])
sma50, sma200 = prices[-50:].mean(), prices[-200:].mean()
regime = "Tren menguat" if prices[-1] > sma200 and sma50 > sma200 else "Tren melemah" if prices[-1] < sma200 and sma50 < sma200 else "Tren campuran"
returns = prices[1:] / prices[:-1] - 1
vol = np.std(returns[-20:], ddof=1) * np.sqrt(252)
flow = bundle["foreign_flow"]
valuation = bundle["valuation_readiness"]
print("Source:", bundle_path, "Breadth date:", dates[-1], "Coverage:", len(changes), "/", valuation["universe_count"])
print("Above SMA50:", sum(above50), "/", len(above50), "Regime:", regime, "Vol p.a.:", vol)
print("Report dates:", sorted({r["price_as_of"] for r in valuation["forward_pe_by_issuer"] if r["price_as_of"]}))
fig, axes = plt.subplots(2, 2, figsize=(11, 6), constrained_layout=True)
ax = axes[0, 0]
items = flow["unusual_foreign_flow"][:3]
ax.barh([r["ticker"] for r in items], [r["z20"] for r in items], color=["#49c99e" if r["z20"] > 0 else "#e4777b" for r in items])
ax.axvline(0, color="gray"); ax.set_xlim(-3, 3); ax.set_title(f"Unusual foreign flow: {len(flow['unusual_foreign_flow'])} issuers"); ax.set_xlabel("Z versus previous 20 sessions; scale ±3")
ax = axes[0, 1]
ax.barh(["IHSG close", "SMA50", "SMA200"], [prices[-1], sma50, sma200], color="#62bcf7")
ax.set_title(f"{regime} | vol20 {vol:.1%} p.a."); ax.set_xlabel("Index points")
ax = axes[1, 0]
ax.bar(["Naik", "Tetap", "Turun"], counts, color=["#49c99e", "#8192a7", "#e4777b"])
ax.set_title(f"Breadth LQ45: {dates[-1]}"); ax.set_ylabel("Issuer count")
ax = axes[1, 1]
ax.barh(["LQ45 snapshot"], [valuation["market_cap_idr"] / 1e12], color="#62bcf7")
ax.set_title(f"Market cap: {valuation['market_cap_coverage']}/{valuation['universe_count']} coverage")
ax.set_xlabel("Rp trillion; not whole IDX")
plt.show()
print("Descriptive snapshots only: no forecast, model training, or broker ownership inference.")
